# Day 28: Data Quality and Schema Diagnostics

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Report row-level schema problems, missing values and duplicate records without changing user data.

## Theory

Validation decides whether prediction is allowed. A data-quality report helps users locate the rows they must correct. Duplicate records are reported, not silently removed.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Inspect the unmodified historical evaluation inputs.

In [2]:
from src.dataset import original_split
from src.data_quality import inspect_batch
import pandas as pd
X=original_split()[1]
report=inspect_batch(X)
assert report["invalid_rows"]==0
print(report)

{'rows': 130, 'valid_rows': 130, 'invalid_rows': 0, 'duplicate_rows': 0, 'missing_cells': 0, 'issues': []}


Exercise malformed scenarios and save diagnostic evidence.

In [3]:
bad=X.head(3).copy().astype(object)
bad.iloc[1,bad.columns.get_loc("age")]=None
bad.iloc[2,bad.columns.get_loc("school")]="BAD"
errors=inspect_batch(bad)
assert errors["invalid_rows"]==2
print(json.dumps(errors,indent=2))
Path("reports/data_quality.json").write_text(json.dumps({"reference":report,"explicit_test_scenario":errors},indent=2))

{
  "rows": 3,
  "valid_rows": 1,
  "invalid_rows": 2,
  "duplicate_rows": 0,
  "missing_cells": 1,
  "issues": [
    {
      "row": 2,
      "issue": "Row 2, age: expected an integer between 15 and 22"
    },
    {
      "row": 3,
      "issue": "Row 3, school: expected one of ['GP', 'MS']"
    }
  ]
}


519

Run row-position, missing-data and duplicate-data checks.

In [4]:
r=subprocess.run([sys.executable,"-m","pytest","-q","tests/test_data_quality.py"],capture_output=True,text=True)
print(r.stdout)
assert r.returncode==0,r.stderr

...                                                                      [100%]
3 passed in 1.43s



## Findings and interpretation

The reference records conform to the serving schema. Explicit malformed test cases return row locations and correction guidance without echoing raw input values.

## Limitations

Completeness and domain validity cannot establish factual correctness or educational suitability. Duplicate students cannot be identified without identifiers, which this project does not collect.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Report row-level schema problems, missing values and duplicate records without changing user data.

## Next-day plan

Expose the shared prediction service through a local API.